# 03 - Tree models: LightGBM, XGBoost, CatBoost

**Goal:** beat the baseline with "gradient boosted trees". These build many small decision trees,
each one fixing the mistakes of the ones before. They are usually the best models for table data.

- Same 5 folds as notebook 02.
- **Early stopping:** each fold stops adding trees when the score on its held-out part stops improving.
- **Optuna** (section at the end) searches for better LightGBM settings. It is capped by time so it can't run for hours.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from config import *
from utils import *
from train import train_lgbm, train_xgb, train_catboost

In [2]:
train, test, sample = load_data()
folds = get_folds(train)
y = train[TARGET].values
X = to_category(train[FEATURES])
X_test = to_category(test[FEATURES])

## LightGBM

In [3]:
oof, test_pred = train_lgbm(X, y, X_test, folds)
auc, ll = score(y, oof)
print(f"LightGBM CV  AUC = {auc:.5f}   logloss = {ll:.5f}")
save_preds("lgbm_v1", oof, test_pred)
log_experiment("lgbm_v1", "raw", auc, ll, "default params")
make_submission(test_pred, "lgbm_v1", sample);

  fold 0: AUC = 0.94056


  fold 1: AUC = 0.94149


  fold 2: AUC = 0.94285


  fold 3: AUC = 0.94223


  fold 4: AUC = 0.94177


LightGBM CV  AUC = 0.94177   logloss = 0.22657


Saved lgbm_v1.csv  rows=286571


## XGBoost

In [4]:
oof, test_pred = train_xgb(X, y, X_test, folds)
auc, ll = score(y, oof)
print(f"XGBoost CV  AUC = {auc:.5f}   logloss = {ll:.5f}")
save_preds("xgb_v1", oof, test_pred)
log_experiment("xgb_v1", "raw", auc, ll, "default params")
make_submission(test_pred, "xgb_v1", sample);

  fold 0: AUC = 0.94070


  fold 1: AUC = 0.94147


  fold 2: AUC = 0.94295


  fold 3: AUC = 0.94239


  fold 4: AUC = 0.94182


XGBoost CV  AUC = 0.94186   logloss = 0.22643


Saved xgb_v1.csv  rows=286571


## CatBoost
CatBoost handles text columns on its own, so we pass the raw text (no `category` type needed).

In [5]:
oof, test_pred = train_catboost(train[FEATURES], y, test[FEATURES], folds, cat_cols=CAT_COLS)
auc, ll = score(y, oof)
print(f"CatBoost CV  AUC = {auc:.5f}   logloss = {ll:.5f}")
save_preds("cat_v1", oof, test_pred)
log_experiment("cat_v1", "raw", auc, ll, "default params")
make_submission(test_pred, "cat_v1", sample);

  fold 0: AUC = 0.94064


  fold 1: AUC = 0.94145


  fold 2: AUC = 0.94271


  fold 3: AUC = 0.94238


  fold 4: AUC = 0.94171


CatBoost CV  AUC = 0.94177   logloss = 0.22659


Saved cat_v1.csv  rows=286571


## Optuna: tune LightGBM settings
To keep it fast, Optuna tests settings on **fold 0 only**, with a time limit.
Then we retrain the best settings on all 5 folds.
Raise `OPTUNA_MINUTES` if you have more time.

In [6]:
import optuna
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
optuna.logging.set_verbosity(optuna.logging.WARNING)

OPTUNA_MINUTES = 15
tr, va = folds != 0, folds == 0

def objective(trial):
    p = {
        "learning_rate": 0.05,
        "num_leaves": trial.suggest_int("num_leaves", 15, 255, log=True),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 500, log=True),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.4, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.5, 1.0),
        "bagging_freq": 1,
        "lambda_l1": trial.suggest_float("lambda_l1", 1e-3, 10, log=True),
        "lambda_l2": trial.suggest_float("lambda_l2", 1e-3, 10, log=True),
        "max_bin": trial.suggest_categorical("max_bin", [63, 255, 511]),
        "cat_smooth": trial.suggest_float("cat_smooth", 1, 100, log=True),
        "n_estimators": 3000, "verbose": -1, "random_state": SEED, "n_jobs": -1,
    }
    m = lgb.LGBMClassifier(**p).fit(X[tr], y[tr], eval_set=[(X[va], y[va])], eval_metric="auc",
                                    callbacks=[lgb.early_stopping(100, verbose=False)])
    return roc_auc_score(y[va], m.predict_proba(X[va])[:, 1])

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(objective, timeout=OPTUNA_MINUTES * 60)
print(f"Trials run: {len(study.trials)}   best fold-0 AUC: {study.best_value:.5f}")
best_params = study.best_params
best_params

Trials run: 46   best fold-0 AUC: 0.94178


{'num_leaves': 25,
 'min_child_samples': 186,
 'feature_fraction': 0.40542747440682847,
 'bagging_fraction': 0.968336511497273,
 'lambda_l1': 0.00368464480823872,
 'lambda_l2': 0.3527941199754003,
 'max_bin': 511,
 'cat_smooth': 3.282184184531182}

In [7]:
# Retrain the best settings on all 5 folds, with a smaller learning rate for a little extra accuracy
oof, test_pred = train_lgbm(X, y, X_test, folds, params={**best_params, "learning_rate": 0.02})
auc, ll = score(y, oof)
print(f"LightGBM (Optuna) CV  AUC = {auc:.5f}   logloss = {ll:.5f}")
save_preds("lgbm_optuna", oof, test_pred)
log_experiment("lgbm_optuna", "raw", auc, ll, f"optuna {len(study.trials)} trials, lr 0.02")
make_submission(test_pred, "lgbm_optuna", sample)

import json
(ROOT / "best_lgbm_params.json").write_text(json.dumps(best_params, indent=2))

  fold 0: AUC = 0.94178


  fold 1: AUC = 0.94254


  fold 2: AUC = 0.94383


  fold 3: AUC = 0.94326


  fold 4: AUC = 0.94277


LightGBM (Optuna) CV  AUC = 0.94283   logloss = 0.22468


Saved lgbm_optuna.csv  rows=286571


258

In [8]:
pd.read_csv(EXPERIMENTS_FILE).sort_values("cv_auc", ascending=False)

,model,features,cv_auc,cv_logloss,notes,date
4,lgbm_optuna,raw,0.942828,0.224683,"optuna 46 trials, lr 0.02",2026-10-17 15:59
2,xgb_v1,raw,0.941857,0.226431,default params,2026-10-17 15:11
3,cat_v1,raw,0.941772,0.226594,default params,2026-10-17 15:39
1,lgbm_v1,raw,0.941771,0.226571,default params,2026-10-17 15:09
0,logreg_v1,raw,0.938094,0.233295,one-hot + scaling,2026-10-17 15:07
